In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
import matplotlib as mpl  # Импортируем основной модуль matplotlib

# Настройка фигуры и 3D осей
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
ax.set_facecolor('black')

# Параметры сетки
x = np.linspace(-5, 5, 100)
y = np.linspace(-5, 5, 100)
X, Y = np.meshgrid(x, y)
Z = np.zeros_like(X)

# Инициализация поверхности
surf = ax.plot_surface(X, Y, Z, cmap='viridis', rstride=1, cstride=1, 
                       alpha=0.8, antialiased=True, linewidth=0)

# Настройка визуализации
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Волны от падающей капли', fontsize=14)  # Обновляем заголовок
ax.set_zlim(-3, 3)
fig.colorbar(surf, shrink=0.5, aspect=10)

# Функция генерации поверхности (модифицирована)
def generate_surface(frame):
    time = frame * 0.1
    # Расстояние от центра падения капли (0,0)
    R = np.sqrt(X**2 + Y**2)
    # Параметры волны: амплитуда, волновое число, затухание
    A = 1.5
    k = 2.0
    damping = 0.1
    return A * np.sin(k * R - time) * np.exp(-damping * R)  # Расходящаяся волна

# Функция обновления кадра (без изменений)
def update(frame):
    global surf
    
    # Удаляем предыдущую поверхность
    if surf:
        surf.remove()
    
    # Генерируем новую поверхность
    Z = generate_surface(frame)
    
    # ИСПРАВЛЕНИЕ: используем новый синтаксис для цветовых карт
    plasma_cmap = mpl.colormaps['plasma'].resampled(200)
    
    # Создаем новую поверхность с динамической цветовой картой
    surf = ax.plot_surface(X, Y, Z, cmap=plasma_cmap, 
                          rstride=2, cstride=2, 
                          alpha=0.9, 
                          linewidth=0.1,
                          antialiased=True)
    
    # Плавно меняем угол обзора
    ax.view_init(elev=30, azim=frame*0.5)
    
    # Обновляем заголовок
    ax.set_title(f'Расходящиеся волны: Кадр {frame}', fontsize=14)
    
    return [surf]

# Создание анимации
ani = FuncAnimation(fig, update, frames=200, interval=50, blit=False)

plt.tight_layout()
plt.show()

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Polygon

# Настройка параметров сигнала
height = 0.8  # Высота прямоугольного импульса
duration = 1  # Длительность импульса
fps = 20  # Кадров в секунду

# Создаем временную ось
t = np.linspace(-1.5, 2.5, 1000)

# Функция прямоугольного сигнала
def rect_signal(x, height, start, end):
    return np.where((x >= start) & (x <= end), height, 0)

# Создаем фигуру и оси
fig, ax = plt.subplots(figsize=(10, 6), dpi=100)
plt.subplots_adjust(bottom=0.15)
ax.set_xlim(-1.5, 2.5)
ax.set_ylim(0, 1.5)
ax.set_xlabel('Время')
ax.set_ylabel('Амплитуда')
ax.grid(True, linestyle='--', alpha=0.7)
ax.set_title('Свертка прямоугольного сигнала с самим собой', fontsize=14)

# Основные элементы анимации
original_signal, = ax.plot(t, rect_signal(t, height, 0, duration), 'b-', linewidth=2, label='Исходный сигнал')
shifted_signal, = ax.plot([], [], 'orange', linewidth=2, label='Сдвинутая копия')
convolution_line, = ax.plot([], [], 'r-', linewidth=3, label='Результат свертки')
area_label = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=12, 
                     bbox=dict(facecolor='white', alpha=0.8))
time_label = ax.text(0.02, 0.85, '', transform=ax.transAxes, fontsize=12,
                     bbox=dict(facecolor='white', alpha=0.8))

# Создаем пустой полигон для области произведения
product_poly = Polygon(np.empty((0, 2)), closed=True, facecolor='green', alpha=0.4)
ax.add_patch(product_poly)

# Вычисление аналитической свертки
def analytical_convolution(tau):
    result = np.zeros_like(tau)
    for i, t_val in enumerate(tau):
        if 0 <= t_val < 1:
            result[i] = height**2 * t_val
        elif 1 <= t_val <= 2:
            result[i] = height**2 * (2 - t_val)
    return result

conv_result = analytical_convolution(t)

# Инициализация
def init():
    shifted_signal.set_data([], [])
    convolution_line.set_data([], [])
    product_poly.set_xy(np.empty((0, 2)))
    time_label.set_text('')
    area_label.set_text('')
    return original_signal, shifted_signal, convolution_line, product_poly, time_label, area_label

# Функция обновления кадра
def update(frame):
    # Сдвинутая копия сигнала
    shifted = rect_signal(t, height, frame - duration, frame)
    shifted_signal.set_data(t, shifted)
    
    # Находим область пересечения прямоугольников
    overlap_start = max(0, frame - duration)
    overlap_end = min(duration, frame)
    
    # Создаем вершины для полигона пересечения
    if overlap_start < overlap_end:
        # Координаты области пересечения
        x_vals = [overlap_start, overlap_end, overlap_end, overlap_start]
        y_vals = [0, 0, height, height]
        
        # Создаем массив вершин: [x1, y1], [x2, y2], ...
        vertices = np.column_stack([x_vals, y_vals])
    else:
        # Если пересечения нет - пустой полигон
        vertices = np.empty((0, 2))
    
    product_poly.set_xy(vertices)
    
    # Результат свертки (накопленный)
    conv_display = np.where(t <= frame, conv_result, np.nan)
    convolution_line.set_data(t, conv_display)
    
    # Вычисление текущей площади (только в области пересечения)
    if overlap_start < overlap_end:
        current_area = height**2 * (overlap_end - overlap_start)
    else:
        current_area = 0.0
    
    area_label.set_text(f'Площадь: {current_area:.3f}')
    time_label.set_text(f't = {frame:.2f}')
    
    return original_signal, shifted_signal, convolution_line, product_poly, time_label, area_label

# Создание анимации
frames = np.linspace(-0.5, 2.5, 100)
ani = FuncAnimation(fig, update, frames=frames, init_func=init, blit=True, interval=1000/fps)

# Добавление легенды
ax.legend(loc='upper right', framealpha=0.9)

# Сохранение в GIF
print("Сохранение анимации...")
ani.save('convolution_animation.gif', writer='pillow', fps=fps, dpi=100)
print("Анимация успешно сохранена как 'convolution_animation.gif'")

# Явное сохранение ссылки на анимацию до завершения работы
plt.close(fig)
del ani

Сохранение анимации...
Анимация успешно сохранена как 'convolution_animation.gif'
